# C3 — Deep tabular models: MLP, TabNet, FT-Transformer (H4)

Inputs: C1-R features with median imputation + missing indicators + standard scaling; unweighted BCE; no early stopping (epochs searched). Search on the v2 fold (MLP 20 configs, TabNet 10, FT-Transformer 8 — budgets fixed in advance from a speed-only benchmark), selection by validation PR-AUC, 10 seeds, test once. Design: Study C addendum 2 (commit 59bd4b1).

**H4 (registered: deep models do NOT beat GBMs):** best-validation deep model vs C1-R XGBoost, paired bootstrap of PR-AUC; supported if the CI of deep − GBM is not entirely above 0.

Reads saved outputs; full run ≈ 2.5 h on 5 CPU threads (FT-Transformer ≈ 8 min per fit): `python -m src.models.c3_deep --stage tune` then `--stage final`. Note: torch must be imported before LightGBM/XGBoost on macOS (two OpenMP runtimes otherwise segfault).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

## Validation search (v2 fold)

In [2]:
t = pd.read_csv("outputs/c3_tuning.csv"); t.groupby("model").val_pr_auc.describe().round(4)

,count,mean,std,min,25%,50%,75%,max
model,,,,,,,,
ft_transformer,8.0,0.0178,0.0297,0.0044,0.0048,0.0071,0.0104,0.0908
mlp,20.0,0.0491,0.0163,0.0036,0.0378,0.0560,0.0595,0.0696
tabnet,10.0,0.0255,0.0156,0.0080,0.0135,0.0229,0.0344,0.0585


In [3]:
t.sort_values("val_pr_auc", ascending=False).head(8)[["model", "config_id", "val_pr_auc", "seconds", "config"]]

,model,config_id,val_pr_auc,seconds,config
33,ft_transformer,3,0.090757,349.2,"{""d_token"": 64, ""blocks"": 2, ""attention_dropou..."
10,mlp,10,0.069626,2.4,"{""layers"": 1, ""width"": 256, ""dropout"": 0.06814..."
19,mlp,19,0.067071,10.4,"{""layers"": 3, ""width"": 256, ""dropout"": 0.27961..."
7,mlp,7,0.065008,2.5,"{""layers"": 2, ""width"": 256, ""dropout"": 0.21644..."
18,mlp,18,0.064179,1.8,"{""layers"": 2, ""width"": 128, ""dropout"": 0.15879..."
2,mlp,2,0.059653,4.0,"{""layers"": 2, ""width"": 256, ""dropout"": 0.24475..."
9,mlp,9,0.059427,1.5,"{""layers"": 2, ""width"": 64, ""dropout"": 0.178290..."
24,tabnet,4,0.058494,30.2,"{""n_d"": 16, ""n_a"": 16, ""n_steps"": 3, ""gamma"": ..."


FT-Transformer is the best deep model on validation only because of **one** configuration (0.091); its other seven were 0.004–0.011. Selection is by the registered rule regardless.

## Test results (10 seeds, mean ± std)

In [4]:
pd.read_csv("outputs/c3_summary.csv", index_col=0)

,pr_auc,precision_at_10,precision_at_100,ndcg_at_100,mrr,roc_auc,brier,ece
model,,,,,,,,
ft_transformer,0.021 ± 0.0017,0.26 ± 0.049,0.117 ± 0.019,0.1442 ± 0.0124,0.425 ± 0.2155,0.8123 ± 0.0105,0.0014 ± 0.0,0.0007 ± 0.0004
mlp,0.0175 ± 0.0011,0.24 ± 0.0663,0.128 ± 0.0087,0.1432 ± 0.0082,0.1903 ± 0.0432,0.7602 ± 0.0054,0.0014 ± 0.0,0.0005 ± 0.0002
tabnet,0.0166 ± 0.0034,0.24 ± 0.12,0.11 ± 0.0179,0.1304 ± 0.0303,0.3546 ± 0.333,0.7612 ± 0.0155,0.0014 ± 0.0,0.0007 ± 0.0007


## H4 and secondary comparisons (paired bootstrap, seed-averaged scores)

In [5]:
pd.read_csv("outputs/c3_hypothesis_tests.csv", index_col=0).round(4)

,pr_auc_diff,pr_auc_ci_low,pr_auc_ci_high,pr_auc_p_not_better,p_at_100_diff,p_at_100_ci_low,p_at_100_ci_high,p_at_100_p_not_better
H4_ft_transformer_vs_xgboost,0.0003,-0.0043,0.0087,0.450,-0.02,-0.04,0.0200,0.890
secondary_mlp_vs_xgboost,-0.0043,-0.0108,0.0017,0.930,-0.01,-0.04,0.0400,0.666
secondary_mlp_vs_logreg,-0.0056,-0.0118,-0.0014,0.996,0.01,-0.03,0.0400,0.468
secondary_mlp_vs_epss_pit,0.0004,-0.0061,0.0072,0.490,0.02,-0.02,0.0600,0.304
secondary_tabnet_vs_xgboost,-0.0010,-0.0048,0.0037,0.678,0.02,-0.01,0.0700,0.164
secondary_tabnet_vs_logreg,-0.0023,-0.0080,0.0023,0.856,0.04,0.00,0.0800,0.036
secondary_tabnet_vs_epss_pit,0.0037,-0.0012,0.0095,0.068,0.05,0.01,0.0952,0.016
secondary_ft_transformer_vs_logreg,-0.0010,-0.0053,0.0058,0.630,0.00,-0.03,0.0300,0.704
secondary_ft_transformer_vs_epss_pit,0.0050,0.0001,0.0118,0.020,0.01,-0.04,0.0552,0.476


**H4 supported, in its weak (non-superiority) form.** FT-Transformer − XGBoost PR-AUC = +0.0003 [−0.0043, 0.0087]: no evidence that the deep model is better — but the GBM is not significantly better either, so this is "no difference detected", not "GBMs win". FT-Transformer is also the noisiest model across seeds (PR-AUC ± 0.0017 vs ± 0.0002 for XGBoost) and needs ~100× the compute. MLP and TabNet are numerically below XGBoost and LR; MLP is significantly below LR (−0.0056 [−0.0118, −0.0014]). FT-Transformer is marginally above point-in-time EPSS on PR-AUC (+0.0050 [0.0001, 0.0118], secondary), not on P@100.